<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW17 · Session 1 · Final Project Workshop: End-to-End Analysis

        **Course level:** developing beginner  
        **Prior learning:** all TW9–TW16 statistics, importing, cleaning, visualisation, modelling, clustering, and time-series skills  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Teams build a reproducible business case study from import through modelling. This workshop provides a project contract, code structure, evidence checklist, and peer-review process.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - define a decision-led project question and success criteria
- assign transparent team roles and version responsibilities
- build a reproducible import-clean-analyse-visualise-model workflow
- evaluate both data quality and model evidence
- prepare a concise project report with limitations and next actions


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


## Final project brief

Use one supplied dataset or a lecturer-approved alternative to answer a specific business question. The final workflow must include:

1. import and provenance;
2. quality profile and cleaning log;
3. exploratory statistics;
4. at least two purposeful visualisations;
5. feature engineering;
6. one justified modelling or segmentation method;
7. appropriate validation;
8. limitations, ethics, and a feasible recommendation.

The objective is not the largest model. It is the clearest defensible analysis.


### Create a project contract

Agree the decision, audience, unit of analysis, target or outcome, scope, and success criteria before writing model code.


In [ ]:
import numpy as np
import pandas as pd

project_contract = {
    "business_question": "Which customer characteristics are associated with churn?",
    "decision_supported": "Prioritise retention outreach for further testing",
    "audience": "Customer-retention manager",
    "unit_of_analysis": "One synthetic customer",
    "outcome": "churn",
    "success_criteria": [
        "Reproducible notebook",
        "Improves on baseline F1",
        "Recall and precision both reported",
        "Limitations and subgroup checks included",
    ],
    "out_of_scope": [
        "Automated customer decisions",
        "Causal claims",
        "Use on real people without governance review",
    ],
}
print(project_contract)


## 1. Team workflow

Suggested roles rotate rather than create silos:

- project lead: scope and integration;
- data steward: provenance, schema, and quality;
- analyst: EDA and statistical reasoning;
- modeller: pipeline and validation;
- reviewer/presenter: challenge assumptions and communicate evidence.

Every member should understand the full notebook and be able to explain one decision outside their main role.


### Record roles and review ownership

A responsibility table prevents important checks from becoming nobody's task.


In [ ]:
responsibilities = pd.DataFrame({
    "work_item": [
        "Project contract", "Import and data dictionary", "Cleaning log",
        "EDA and charts", "Model and validation", "Ethics review",
        "Presentation rehearsal",
    ],
    "owner": ["Lead", "Data steward", "Data steward", "Analyst", "Modeller", "Reviewer", "All"],
    "reviewer": ["All", "Reviewer", "Analyst", "Reviewer", "Lead", "All", "All"],
})
print(responsibilities)


### Load the selected dataset and capture provenance

The supplied churn dataset is synthetic and contains no personal data.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

source_path = DATA_DIR / "customer_churn.csv"
project_data = pd.read_csv(source_path)
provenance = {
    "file": source_path.name,
    "rows": len(project_data),
    "columns": project_data.shape[1],
    "source": "Synthetic course dataset",
    "loaded_with": "pandas.read_csv",
}
print(provenance)


## 2. Quality gate

The project cannot proceed until required columns, target values, key uniqueness, missingness, numerical ranges, and category vocabularies are checked.


### Run the quality gate

Failures should stop the workflow or be logged and repaired before modelling.


In [ ]:
required = {
    "customer_id", "age", "tenure_months", "monthly_spend",
    "support_tickets", "contract_type", "autopay", "churn",
}
quality_gate = {
    "missing_required_columns": sorted(required - set(project_data.columns)),
    "duplicate_customer_ids": int(project_data["customer_id"].duplicated().sum()),
    "missing_cells": int(project_data.isna().sum().sum()),
    "invalid_target": int((~project_data["churn"].isin([0, 1])).sum()),
    "invalid_age": int((~project_data["age"].between(18, 100)).sum()),
    "invalid_spend": int((project_data["monthly_spend"] < 0).sum()),
}
print(pd.Series(quality_gate))
assert not quality_gate["missing_required_columns"]
assert quality_gate["duplicate_customer_ids"] == 0
assert quality_gate["invalid_target"] == 0


## 3. Exploratory evidence

EDA should describe the target balance, feature distributions, group differences, and unusual observations without repeatedly testing until a desirable result appears.


### Create core descriptive evidence

The table compares retained and churn groups without implying causation.


In [ ]:
target_balance = project_data["churn"].value_counts().rename(index={0: "Retained", 1: "Churned"})
group_summary = project_data.groupby("churn").agg(
    customers=("customer_id", "count"),
    median_tenure=("tenure_months", "median"),
    mean_spend=("monthly_spend", "mean"),
    mean_tickets=("support_tickets", "mean"),
)
print(target_balance)
print(group_summary.round(2))


### Create two purposeful visualisations

One shows target balance; the other compares monthly-spend distributions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(target_balance.index, target_balance.values, color=["#2563eb", "#d97706"])
axes[0].set_title("Customer outcome counts")
axes[0].set_ylabel("Customers")

axes[1].boxplot(
    [
        project_data.loc[project_data["churn"] == 0, "monthly_spend"],
        project_data.loc[project_data["churn"] == 1, "monthly_spend"],
    ],
    labels=["Retained", "Churned"],
    patch_artist=True,
)
axes[1].set_title("Monthly spend by observed outcome")
axes[1].set_ylabel("Monthly spend (£)")

fig.suptitle("Churn case-study exploratory evidence")
fig.tight_layout()
plt.show()
plt.close(fig)


## 4. Reproducible model pipeline

The final test set is created once. Preprocessing and model fitting occur only on training data. The baseline is evaluated with the same test rows.


### Prepare a stratified split

customer_id remains available for traceability but is not a predictor.


In [ ]:
X = project_data.drop(columns=["customer_id", "churn"])
y = project_data["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)
print(X_train.shape, X_test.shape)


### Build and fit baseline plus logistic pipeline

All learned preprocessing is contained inside the candidate pipeline.


In [ ]:
numeric_columns = ["age", "tenure_months", "monthly_spend", "support_tickets"]
categorical_columns = ["contract_type", "autopay"]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_columns),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_columns),
])
candidate = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
baseline = DummyClassifier(strategy="most_frequent")

candidate.fit(X_train, y_train)
baseline.fit(X_train, y_train)
print("Models fitted.")


### Evaluate against baseline

The comparison includes threshold metrics and ranking performance.


In [ ]:
evaluation_rows = []
for name, model in {"Baseline": baseline, "Candidate": candidate}.items():
    predictions = model.predict(X_test)
    probabilities = (
        model.predict_proba(X_test)[:, 1]
        if hasattr(model, "predict_proba")
        else predictions
    )
    evaluation_rows.append({
        "model": name,
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
    })
evaluation = pd.DataFrame(evaluation_rows).set_index("model")
print(evaluation.round(3))


### Create an error table

Error counts connect metrics to operational consequences.


In [ ]:
project_predictions = candidate.predict(X_test)
matrix = confusion_matrix(y_test, project_predictions)
error_table = pd.DataFrame(
    matrix,
    index=["Actual retained", "Actual churn"],
    columns=["Predicted retained", "Predicted churn"],
)
print(error_table)
print(classification_report(y_test, project_predictions, zero_division=0))


## 5. Ethics and limitations

Discuss representativeness, missing variables, proxy discrimination, unequal error rates, privacy, human oversight, monitoring, and the consequence of intervention. A model score should inform a reviewed process, not automatically determine treatment.


### Create a risk register

Every risk has evidence to collect and a mitigation rather than a vague warning.


In [ ]:
risk_register = pd.DataFrame([
    {
        "risk": "Synthetic data does not represent real customers",
        "evidence_needed": "External validation on governed real data",
        "mitigation": "Do not deploy this teaching model",
    },
    {
        "risk": "Unequal false-negative rates",
        "evidence_needed": "Subgroup recall and sample-size review",
        "mitigation": "Threshold and policy review with affected stakeholders",
    },
    {
        "risk": "Intervention may annoy retained customers",
        "evidence_needed": "Controlled pilot and opt-out monitoring",
        "mitigation": "Human review and low-risk outreach",
    },
])
print(risk_register)


## Project evidence checklist

The submission should run from a clean kernel, contain no unexplained errors, distinguish raw and transformed data, justify every metric and chart, and finish with a decision, limitation, and next test.


### Create a self-audit scorecard

Teams replace False only after attaching evidence.


In [ ]:
project_checks = pd.Series({
    "question_is_decision_led": True,
    "source_and_schema_recorded": True,
    "raw_data_preserved": True,
    "cleaning_decisions_logged": False,
    "charts_answer_questions": True,
    "baseline_compared": True,
    "test_set_untouched_until_final": True,
    "limitations_and_ethics_reviewed": True,
    "notebook_runs_top_to_bottom": False,
})
print(project_checks)
print("Completion:", f"{project_checks.mean():.0%}")


## Student project sprint

These five deliverables become the backbone of the group submission.


        ### Student task 1: Approve the project contract

        <div class="task"><strong>Your job:</strong> Complete question, decision, audience, unit, target, success criteria, exclusions, and team role assignments.</div>

        **Check your work**

        - The question is answerable with available data.
- Success includes analytic and business criteria.
- Excluded uses prevent overreach.


In [ ]:
# STUDENT TASK 1
team_contract = {
    "question": "",
    "decision": "",
    "audience": "",
    "unit": "",
    "target_or_outcome": "",
    "success_criteria": [],
    "excluded_uses": [],
    "roles": {},
}
print(team_contract)


        ### Student task 2: Build the data-quality gate

        <div class="task"><strong>Your job:</strong> Create schema, key, missingness, range, and category checks. Stop with helpful assertions when a critical rule fails.</div>

        **Check your work**

        - Checks are calculated from source data.
- Critical and warning-level issues are distinguished.
- Every assertion has a useful message.


In [ ]:
# STUDENT TASK 2
team_quality_gate = {}
print(pd.Series(team_quality_gate))


        ### Student task 3: Produce EDA evidence

        <div class="task"><strong>Your job:</strong> Create one overall summary, one grouped table, and two complementary charts tied to the project question.</div>

        **Check your work**

        - Every output answers a named question.
- Counts and uncertainty context are visible.
- No causal claim is made from association.


In [ ]:
# STUDENT TASK 3
team_summary = pd.Series(dtype=float)
team_grouped = pd.DataFrame()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# Add the evidence.
plt.close(fig)
print(team_summary)
print(team_grouped)


        ### Student task 4: Build and evaluate a pipeline

        <div class="task"><strong>Your job:</strong> Compare an appropriate baseline and one justified candidate using a leakage-safe split or time-aware validation.</div>

        **Check your work**

        - Prediction time and feature availability are stated.
- Preprocessing fits on training data only.
- Metrics match error costs and include baseline.


In [ ]:
# STUDENT TASK 4
team_model_results = pd.DataFrame()
print(team_model_results)


        ### Student task 5: Write the executive conclusion

        <div class="task"><strong>Your job:</strong> Create strings for finding, evidence, limitation, recommendation, validation next step, and ethical safeguard.</div>

        **Check your work**

        - Every claim points to output.
- Recommendation strength matches evidence.
- A feasible next validation step and safeguard are included.



**Optional extension:** Exchange notebooks with another team and resolve every reproducibility issue they find.


In [ ]:
# STUDENT TASK 5
executive_conclusion = {
    "finding": "",
    "evidence": "",
    "limitation": "",
    "recommendation": "",
    "next_validation": "",
    "ethical_safeguard": "",
}
print(executive_conclusion)


        ## Knowledge check

        Answer these without running new code first.

        1. What makes a project question decision-led?
2. Why rotate or review team roles?
3. What is a data-quality gate?
4. Why compare with a baseline?
5. What separates a finding from a recommendation?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Session 2 rehearses the presentation, runs a final reproducibility audit, and reviews the complete module.
